In [9]:
import os
from astropy.convolution import convolve_fft, Gaussian2DKernel

import scipy.ndimage as nd

import matplotlib.pyplot as plt
import matplotlib.colors as colors
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.ticker as ticker
from matplotlib import colormaps
from matplotlib.colors import ListedColormap
import matplotlib.cm as cm


import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy.visualization import ImageNormalize, SqrtStretch
from sunpy.coordinates.ephemeris import get_body_heliographic_stonyhurst
from astropy.coordinates import solar_system_ephemeris
from astropy.modeling.functional_models import Disk2D

import sunpy.coordinates  as coord # NOQA
import sunpy.map
from sunpy.net import Fido
from sunpy.net import attrs as a
from sunpy.coordinates import frames

from sunkit_image.coalignment import mapsequence_coalign_by_match_template as mc_coalign
from sunkit_image.coalignment import calculate_match_template_shift as mc_shift

import numpy as np

import cmasher as cmr

from identification_utils import *

from datetime import datetime, timedelta
from astropy.io import fits

In [10]:
###Parameters to load data (see sunpy tutorial)

#mail access
jsoc_email = "stucki@ieec.cat"


#data names
Ic_serie ='hmi.Ic_noLimbDark_720s'
M_serie = 'hmi.M_45s'

#directory to retrieve the files into
path = '/home/sophie-stucki/starsim/starsim/SDO_input/SDO_images'
maps_path = '/home/sophie-stucki/starsim/starsim/SDO_input/maps_2021/'

In [11]:
# Downsampling
out_shape = (1024, 1024)

# Noise threshold (from Sen & al. 2023)
noise_thresh = 8

I_th = 0.89

In [12]:
# Set the date range you want to loop over
start_date = datetime.strptime('2021-01-01', '%Y-%m-%d')
end_date = datetime.strptime('2021-01-04', '%Y-%m-%d')  # loop will include up to 2017-08-27
first_day = datetime.strptime('2021-01-01', '%Y-%m-%d')

shift = int((start_date - first_day).days)

day_step = 1

i = shift

days_list = []

print(shift)

0


In [13]:
# Loop day by day
current_date = start_date
while current_date < end_date:


    # Time range for this day
    start_time = current_date.replace(hour=0, minute=0, second=0)
    end_time = start_time + timedelta(seconds=60)

    # Format if needed
    start_str = start_time.strftime('%Y-%m-%dT%H:%M:%S')
    end_str = end_time.strftime('%Y-%m-%dT%H:%M:%S')

    print(f"Processing {start_str} to {end_str}")

    # Put your SunPy/Fido data access or map logic here

    ###Load the data
    date_str = current_date.strftime('%Y%m%d')

    try:
        cont_sequence, los_sequence = load_data(start_time, end_time, jsoc_email, Ic_serie, M_serie, path=path)

        cont_map = cont_sequence.maps[0]
        los_map = los_sequence.maps[0]
    

        RAW_size = cont_map.dimensions

        # downsampling
        cont_map = cont_map.resample(out_shape * u.pix)
        los_map = los_map.resample(out_shape * u.pix)   


        # coordinates
        xg, yg = coord_grid(los_map)

        # remove the noise
        los_sequence_updated = sunpy.map.MapSequence(noise_threshold(los_map, threshold=noise_thresh)) 
        # remove the foreshortening effects
        los_sequence_updated = sunpy.map.MapSequence(removing_foreshortening_effect(los_sequence_updated.maps[0],xg,yg))

        # crop
        min_p = int(np.argwhere(cont_map.data[int(out_shape[0]/2), :] >= 0 ).min() - 1)
        max_p = int(np.argwhere(cont_map.data[int(out_shape[0]/2), :] >= 0 ).max() + 1)


        cont_map = sunpy.map.sources.HMIMap(cont_map.data[min_p:max_p, min_p:max_p], cont_map.fits_header, memmap=True)
        los_map = sunpy.map.sources.HMIMap(np.abs(los_sequence_updated.maps[0].data[min_p:max_p, min_p:max_p]), los_sequence_updated.maps[0].fits_header, memmap=True)

        xg = xg[min_p:max_p, min_p:max_p]
        yg = yg[min_p:max_p, min_p:max_p]

        # plot the new sdo images
        # two_graphs_plot(cont_map, los_map)

        ### identification following Sen & al. 2023

        active_area = active_area_identification(cont_map, los_map, xg, yg, I_th=I_th)
        spot_area, smooth_spot_area, feature_area, smooth_feature_area = active_area_smoothing(active_area, spot_threshold=0.2, spot_kernel_size=2, feature_threshold=0.2, feature_kernel_size=2) 

        area_th = micro_hemisphere_to_arcsec2(los_sequence_updated.maps[0], 20).value
        identification, plage_nbr = network_identification(feature_area, smooth_feature_area, los_map.scale[0].value, area_th, method='scipy')
        nbr, locs, pxl_area = spot_nbr(spot_area, smooth_spot_area)

        # plot the identification map
        # identifiation_plot(identification, spot_area, cont_map, locs, filename=path+'identification_map_downsampling_{}_{}.pdf'.format(int(RAW_size[0].value/out_shape[0]), t))

        ### save the faculae and spot maps for starsim

        facula_map = np.copy(identification)

        facula_map[np.isnan(facula_map)] = 0

        # facula_map = np.zeros(np.shape(identification))
        facula_map[np.isnan(cont_map.data)] = np.nan

        spot_map = np.copy(spot_area)
        # spot_map = np.zeros(np.shape(spot_area))
        
        spot_map[np.isnan(spot_map)] = 0
        spot_map[np.isnan(cont_map.data)] = np.nan

        facula_map = np.flip(facula_map)
        spot_map = np.flip(spot_map)


        
        np.savetxt(maps_path+'faculae_map_{:.1f}.txt'.format(i), facula_map)
        np.savetxt(maps_path+'spot_map_{:.1f}.txt'.format(i), spot_map)
        np.savetxt(maps_path+'cont_map_{:.1f}.txt'.format(i), np.flip(cont_map.data))
        np.savetxt(maps_path+'los_map_{:.1f}.txt'.format(i), np.flip(los_map.data))

        day = (current_date - start_date).days
        days_list.append(day)
    
    except:
        pass

    print('Maps: done')
    
    try:
        os.remove(f"{path}/hmi.ic_nolimbdark_720s.{date_str}_000000_TAI.3.continuum.fits")
    except:
        print("1. Failing deleting the cont. file")
        pass

    try:
        os.remove(f"{path}/hmi.m_45s.{date_str}_000045_TAI.2.magnetogram.fits")
    except:
        print("1. Failing deleting the magneto. file")
        pass



    if os.path.exists(f"{path}/hmi.ic_nolimbdark_720s.{date_str}_000000_TAI.3.continuum.fits"):
        os.remove(f"{path}/hmi.ic_nolimbdark_720s.{date_str}_000000_TAI.3.continuum.fits")
        print("2. Failing deleting the cont. file")
    else:
        print("The cont. file does not exist") 

    if os.path.exists(f"{path}/hmi.m_45s.{date_str}_000045_TAI.2.magnetogram.fits"):
        os.remove(f"{path}/hmi.m_45s.{date_str}_000045_TAI.2.magnetogram.fits")
        print("2. Failing deleting the magneto. file")
    else:
        print("The magneto. file does not exist") 
            
    # two_graphs_plot(cont_map, los_map)

    # Move to the next day
    current_date += timedelta(days=int(day_step))
    i += 1

np.savetxt(maps_path+'days_list.txt', days_list)

Processing 2021-01-01T00:00:00 to 2021-01-01T00:01:00


2025-08-28 15:37:05 - drms - INFO: Export request pending. [id=JSOC_20250828_002790, status=2]
2025-08-28 15:37:05 - drms - INFO: Waiting for 0 seconds...
2025-08-28 15:37:06 - sunpy - INFO: 1 URLs found for download. Full request totaling 16MB


INFO: 1 URLs found for download. Full request totaling 16MB [sunpy.net.jsoc.jsoc]


Files Downloaded: 100%|██████████| 1/1 [00:08<00:00,  8.62s/file]
2025-08-28 15:37:27 - drms - INFO: Export request pending. [id=JSOC_20250828_002823, status=2]
2025-08-28 15:37:27 - drms - INFO: Waiting for 0 seconds...
2025-08-28 15:37:28 - drms - INFO: Export request pending. [id=JSOC_20250828_002823, status=1]
2025-08-28 15:37:28 - drms - INFO: Waiting for 5 seconds...
2025-08-28 15:37:33 - drms - INFO: Export request pending. [id=JSOC_20250828_002823, status=1]
2025-08-28 15:37:33 - drms - INFO: Waiting for 5 seconds...
2025-08-28 15:37:39 - drms - INFO: Export request pending. [id=JSOC_20250828_002823, status=1]
2025-08-28 15:37:39 - drms - INFO: Waiting for 5 seconds...
2025-08-28 15:37:44 - drms - INFO: Export request pending. [id=JSOC_20250828_002823, status=1]
2025-08-28 15:37:44 - drms - INFO: Waiting for 5 seconds...
2025-08-28 15:37:51 - sunpy - INFO: 2 URLs found for download. Full request totaling 32MB


INFO: 2 URLs found for download. Full request totaling 32MB [sunpy.net.jsoc.jsoc]


...

INFO: 1 URLs found for download. Full request totaling 16MB [sunpy.net.jsoc.jsoc]


Files Downloaded: 100%|██████████| 1/1 [00:08<00:00,  8.45s/file]
2025-08-28 15:39:49 - drms - INFO: Export request pending. [id=JSOC_20250828_002831, status=2]
2025-08-28 15:39:49 - drms - INFO: Waiting for 0 seconds...
2025-08-28 15:39:49 - drms - INFO: Export request pending. [id=JSOC_20250828_002831, status=1]
2025-08-28 15:39:49 - drms - INFO: Waiting for 5 seconds...
2025-08-28 15:39:55 - drms - INFO: Export request pending. [id=JSOC_20250828_002831, status=1]
2025-08-28 15:39:55 - drms - INFO: Waiting for 5 seconds...
2025-08-28 15:40:00 - drms - INFO: Export request pending. [id=JSOC_20250828_002831, status=1]
2025-08-28 15:40:00 - drms - INFO: Waiting for 5 seconds...
2025-08-28 15:40:06 - drms - INFO: Export request pending. [id=JSOC_20250828_002831, status=1]
2025-08-28 15:40:06 - drms - INFO: Waiting for 5 seconds...
2025-08-28 15:40:12 - sunpy - INFO: 2 URLs found for download. Full request totaling 32MB


INFO: 2 URLs found for download. Full request totaling 32MB [sunpy.net.jsoc.jsoc]


Files Downloaded: 100%|██████████| 2/2 [00:20<00:00, 10.04s/file]


609
Maps: done
The cont. file does not exist
The magneto. file does not exist


In [18]:
path = "/home/sophie-stucki/starsim/starsim/SDO_input/maps_2021/"

# Go backwards to avoid overwriting
for i in np.arange(365, -1, -1):  # from 364 down to -1
    old_file = os.path.join(path, "spot_map_{:.1f}.txt".format(i))

    if os.path.exists(old_file):
        pass
    else:
        print(f"File does not exist for i={i}")

File does not exist for i=365
